In [ ]:
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from scipy.interpolate import make_smoothing_spline
from scipy.optimize import brentq

### Load Fine Lattice Data

In [ ]:
N = 2
L_f = 80
beta_f = 1.5
beta1_f = 0.0
alpha_f = 0.0
alpha1_f = 0.0
# mod selects the HMC action variant (0 or 1); same mod is used for fine and coarse.
# NOTE: for beta1_f = 0 the fine lattice uses the heatbath sampler, so its data is
#       mod-independent in physics — for mod != 0 you may copy it from the mod0 folder
#       instead of re-simulating.
mod = 1
# obs_fit_type selects the action family: "halfRefVil" (cos(da), no integer s) or
# "RefVil" (full Villain: integer s field, alpha1, + topological Q_s). It routes the
# fine+coarse data into the data_{obs_fit_type}/ tree and the results/ tree.
obs_fit_type = "halfRefVil"

# Data subfolder is keyed by alpha_eff = alpha + alpha1 (NOT alpha/alpha1 separately),
# so the obs beta1-scan and the topo alpha-scan at the same alpha_eff share a subfolder.
alpha_eff_f = alpha_f + alpha1_f
datafile_f = (f"data_{obs_fit_type}/data_PP_corr_N{N}_L{L_f}_mod{mod}/"
              f"N{N}_L{L_f}_beta{beta_f:.3f}_alpha_eff{alpha_eff_f:.3f}/"
              f"PP_corr_N{N}_L{L_f}_beta{beta_f:.3f}_beta1_{beta1_f:.3f}"
              f"_alpha{alpha_f:.3f}_alpha1_{alpha1_f:.3f}.json")

with open(datafile_f, 'r') as f:
    temp = json.load(f)
    argzz_loop_f = temp['argzz_loop']        # dict "{x}x{y}" -> {tau, mean, err} (scalar)
    wilson_loop_f = temp['wilson_loop']      # dict "{x}x{y}" -> {tau:[2], mean:[U,Uz], err:[2]}
    conn_PP_corr_f = np.asarray(temp['conn_PP_corr']['mean'])
    # per-element errors (stored on disk) used to weight the observable fits
    conn_PP_corr_f_err = np.asarray(temp['conn_PP_corr'].get('err', []))
    topo_charge_f = temp.get('topo_charge', {})  # {Q_U,Q_z[, Q_s]} each with "topo_sus"

### Load Coarse Lattice Data

In [ ]:
import glob
import re
from collections import Counter

renorm = 8
L_c = L_f // renorm
renorm_type = 'z'
beta_c = 1.553

# PP-correlation separations (coarse (x,y); fine = (x,y)*renorm). Free choice — PP
# indexes the full conn_PP_corr matrix, so any in-range (x,y) is available. Default
# below replicates the old (0,l) coverage; edit as needed.
pp_xy_list = [[0, l] for l in range(1, 10) if l * 10 <= L_c and l * 10 * renorm <= L_f]

# Auto-discover all coarse data files for this beta_c (and this mod/obs_fit_type).
# Data is grouped into per-(beta, alpha, alpha1) subfolders, so glob across all.
coarse_dir = f"data_{obs_fit_type}/data_PP_corr_N{N}_L{L_c}_mod{mod}"
pattern = f"{coarse_dir}/*/PP_corr_N{N}_L{L_c}_beta{beta_c:.3f}_beta1_*_alpha*_alpha1_*.json"
coarse_files = sorted(glob.glob(pattern))

if len(coarse_files) == 0:
    raise ValueError(f"No data files found for beta_c = {beta_c:.3f} (mod={mod}) in {coarse_dir}")

# Parse beta1, alpha, alpha1 from filenames
parsed = []  # (filepath, beta1, alpha, alpha1)
for fpath in coarse_files:
    fname = os.path.basename(fpath)
    m = re.search(r'beta1_(-?\d+\.\d+)_alpha(-?\d+\.\d+)_alpha1_(-?\d+\.\d+)\.json', fname)
    if m:
        parsed.append((fpath, float(m.group(1)), float(m.group(2)), float(m.group(3))))

if len(parsed) == 0:
    raise ValueError(f"Could not parse beta1/alpha/alpha1 from filenames in {coarse_dir}")

# Auto-detect (alpha_c, alpha1_c) — the most common (alpha, alpha1) pair.
pair_counter = Counter((round(a, 6), round(a1, 6)) for _, _, a, a1 in parsed)
(alpha_c, alpha1_c) = pair_counter.most_common(1)[0][0]
if len(pair_counter) > 1:
    print(f"WARNING: Multiple (alpha, alpha1) pairs found: {sorted(pair_counter)}. "
          f"Using alpha_c = {alpha_c:.3f}, alpha1_c = {alpha1_c:.3f}")

# Filter to files matching (alpha_c, alpha1_c) and collect beta1 values
matching = [(fp, b1) for fp, b1, a, a1 in parsed
           if abs(round(a, 6) - alpha_c) < 1e-6 and abs(round(a1, 6) - alpha1_c) < 1e-6]
matching.sort(key=lambda x: x[1])

data_beta1_c_list = np.array([b1 for _, b1 in matching])

# Load all coarse data. argzz/wilson selection lists are read from the files
# (they record which separations were actually measured).
argzz_loop_c_list = []
wilson_loop_c_list = []
conn_PP_corr_c_list = []
conn_PP_corr_c_err_list = []
topo_charge_c_list = []
argzz_xy_list = wilson_xy_list = None
for fpath, beta1_c in matching:
    with open(fpath, 'r') as f:
        temp = json.load(f)
    argzz_loop_c_list.append(temp['argzz_loop'])
    wilson_loop_c_list.append(temp['wilson_loop'])
    conn_PP_corr_c_list.append(np.asarray(temp['conn_PP_corr']['mean']))
    conn_PP_corr_c_err_list.append(np.asarray(temp['conn_PP_corr'].get('err', [])))
    topo_charge_c_list.append(temp.get('topo_charge', {}))
    if argzz_xy_list is None:
        argzz_xy_list = temp['argzz_xy_list']
        wilson_xy_list = temp['wilson_xy_list']

if len(data_beta1_c_list) == 0:
    raise ValueError("No data files found for the specified beta_c.")

print(f"Found {len(data_beta1_c_list)} coarse data files: beta_c = {beta_c:.3f}, "
      f"alpha_c = {alpha_c:.3f}, alpha1_c = {alpha1_c:.3f}, mod = {mod}, {obs_fit_type}")
print(f"beta1_c range: [{data_beta1_c_list[0]:.3f}, {data_beta1_c_list[-1]:.3f}]")
print(f"pp_xy_list = {pp_xy_list}")
print(f"argzz_xy_list = {argzz_xy_list}  (read from data)")
print(f"wilson_xy_list = {wilson_xy_list}  (read from data)")

In [ ]:
folder_path = (f"results/PP_Wil_beta1_renorm_plot_mod{mod}_{obs_fit_type}/"
               f"N{N}_L{renorm}_{renorm_type}/"
               f"beta{beta_f:.3f}_beta1_{beta1_f:.3f}_alpha{alpha_f:.3f}_alpha1_{alpha1_f:.3f}")
os.makedirs(folder_path, exist_ok=True)
info_dict = {
    'N': N,
    'L_f': L_f,
    'mod': mod,
    'obs_fit_type': obs_fit_type,
    'beta_f': beta_f,
    'beta1_f': beta1_f,
    'alpha_f': alpha_f,
    'alpha1_f': alpha1_f,
    'renorm': renorm,
    'L_c': L_c,
    'renorm_type': renorm_type,
    'beta_c': beta_c,
    'beta1_c_list': data_beta1_c_list.tolist(),
    'alpha_c': alpha_c,
    'alpha1_c': alpha1_c,
    'pp_xy_list_coarse': pp_xy_list,
    'argzz_xy_list_coarse': argzz_xy_list,
    'wilson_xy_list_coarse': wilson_xy_list,
}
file_name = f"renorm_info.json"
file_path = os.path.join(folder_path, file_name)
with open(file_path, 'w') as f:
    json.dump(info_dict, f, indent=4)

In [ ]:
def linear_interpolate(x_vals, y_vals, y_target):
    """Two-point linear interpolation: anchor + fallback for match_coupling."""
    x_vals = np.asarray(x_vals, dtype=float)
    y_vals = np.asarray(y_vals, dtype=float)
    idx = np.argsort(x_vals)
    x_vals = x_vals[idx]
    y_vals = y_vals[idx]
    if y_target < y_vals.min() or y_target > y_vals.max():
        return None
    for i in range(len(y_vals) - 1):
        if (y_vals[i] - y_target) * (y_vals[i+1] - y_target) <= 0:
            t = (y_target - y_vals[i]) / (y_vals[i+1] - y_vals[i])
            return x_vals[i] + t * (x_vals[i+1] - x_vals[i])
    if np.isclose(y_target, y_vals[0]):
        return x_vals[0]
    if np.isclose(y_target, y_vals[-1]):
        return x_vals[-1]
    return None


def _select_root(roots, x_ref, x_lo, x_hi, tol=1e-7):
    cand = []
    for r in np.asarray(roots, dtype=complex).ravel():
        if abs(r.imag) > tol:
            continue
        rv = r.real
        if x_lo - 1e-9 <= rv <= x_hi + 1e-9:
            cand.append(float(rv))
    if not cand:
        return None
    ref = x_ref if x_ref is not None else 0.5 * (x_lo + x_hi)
    return min(cand, key=lambda v: abs(v - ref))


def _make_weights(y_err):
    if y_err is None:
        return None, None
    e = np.asarray(y_err, dtype=float)
    good = np.isfinite(e) & (e > 0)
    if not np.any(good):
        return None, None
    med = float(np.median(e[good]))
    e = np.where(good, e, med)
    return 1.0 / e ** 2, e


def _fit_callable(x, y, w, force_poly=False):
    n = len(x)
    if n < 2:
        return None
    if n >= 5 and not force_poly:
        try:
            return make_smoothing_spline(x, y, w=w)
        except Exception:
            pass
    deg = min(2, n - 2)
    if deg < 1:
        deg = 1
    try:
        return np.poly1d(np.polyfit(x, y, deg, w=w))
    except Exception:
        return None


def _roots_in_range(f, target, x_lo, x_hi, n_grid=1001):
    if f is None:
        return []
    grid = np.linspace(x_lo, x_hi, n_grid)
    try:
        g = np.asarray(f(grid), dtype=float) - target
    except Exception:
        return []
    roots = []
    for i in range(len(g) - 1):
        if g[i] == 0.0:
            roots.append(float(grid[i]))
        elif g[i] * g[i+1] < 0:
            try:
                roots.append(float(brentq(lambda z: float(f(z) - target), grid[i], grid[i+1])))
            except Exception:
                pass
    return roots


def match_coupling(x_vals, y_vals, y_target, y_err=None, y_target_err=None, n_mc=256):
    """Fit y(x) through ALL scan points (error-weighted GCV smoothing spline when
    n >= 5, else a weighted low-order polynomial) and return (x_match, x_err)
    where the fitted curve = y_target. Uses every point and every error bar, so
    one noisy point cannot dominate -- unlike two-point linear interpolation."""
    x = np.asarray(x_vals, dtype=float)
    y = np.asarray(y_vals, dtype=float)
    order = np.argsort(x)
    x = x[order]; y = y[order]
    n = len(x)
    if n < 2:
        return None, None
    y_target = float(y_target)
    w, e_y = _make_weights(np.asarray(y_err, dtype=float)[order] if y_err is not None else None)
    x_lin = linear_interpolate(x, y, y_target)

    def solve(yy, tt):
        roots = _roots_in_range(_fit_callable(x, yy, w), tt, x[0], x[-1])
        if not roots and n >= 5:
            roots = _roots_in_range(_fit_callable(x, yy, w, force_poly=True), tt, x[0], x[-1])
        if not roots:
            return None
        return _select_root(roots, x_lin, x[0], x[-1])

    x_match = solve(y, y_target)
    if x_match is None:
        x_match = x_lin
    if x_match is None:
        return None, None

    x_err = None
    if e_y is not None or y_target_err is not None:
        rng = np.random.default_rng()
        ey = e_y if e_y is not None else np.zeros(n)
        et = float(y_target_err) if y_target_err is not None else 0.0
        samples = []
        for _ in range(n_mc):
            yp = y + ey * rng.standard_normal(n)
            tp = y_target + et * rng.standard_normal()
            xr = solve(yp, tp)
            if xr is not None:
                samples.append(xr)
        if len(samples) >= 2:
            x_err = float(np.std(samples, ddof=1))
    return float(x_match), x_err


def fit_curve(x_vals, y_vals, y_err=None):
    """The same smooth y(x) curve match_coupling inverts, as a callable for plotting."""
    x = np.asarray(x_vals, dtype=float)
    y = np.asarray(y_vals, dtype=float)
    order = np.argsort(x)
    x = x[order]; y = y[order]
    if len(x) < 2:
        return None
    w, _ = _make_weights(np.asarray(y_err, dtype=float)[order] if y_err is not None else None)
    return _fit_callable(x, y, w)


def add_fit_curve(x_vals, y_vals, y_err=None, label="smoothing-spline fit"):
    """Overlay the fitted curve (the one match_coupling uses) on the current axes."""
    f = fit_curve(x_vals, y_vals, y_err)
    if f is None:
        return
    xs = np.linspace(float(np.min(x_vals)), float(np.max(x_vals)), 200)
    plt.plot(xs, np.asarray(f(xs), dtype=float), color="#2ca02c", linewidth=2.2,
             label=label, zorder=2)


def add_coarse_points(x_vals, y_vals, y_err=None, label="Coarse", color=None):
    """Coarse scan as markers with per-point error bars on top of the fitted
    curve (zorder 3 > the spline 2). No connecting line. y_err=None -> points
    only (e.g. corr_len)."""
    plt.errorbar(np.asarray(x_vals, dtype=float), np.asarray(y_vals, dtype=float),
                 yerr=y_err, fmt="o", color=color, capsize=3, label=label, zorder=3)


def _clean_err_list(vals):
    out = []
    for v in vals:
        try:
            fv = float(v)
        except (TypeError, ValueError):
            return None
        if not np.isfinite(fv):
            return None
        out.append(fv)
    return out


def _fmt_match(val, err=None):
    if val is None:
        return "NO MATCH"
    if err is None:
        return f"{val:.4f}"
    return f"{val:.4f} +- {err:.4f}"


# Dictionaries to store matched beta1_c (+ MC errors) from each observable
results_pp = {};        results_pp_err = {}
results_argzz = {};     results_argzz_err = {}
results_wilson = {};    results_wilson_err = {}

### Compare PP-correlation

In [ ]:
for xy in pp_xy_list:
    x, y = int(xy[0]), int(xy[1])
    if not (0 <= x < L_c and 0 <= y < L_c and 0 <= x * renorm < L_f and 0 <= y * renorm < L_f):
        print(f"  PP corr ({x},{y}): out of correlator bounds, skip")
        continue
    ppcorr_c = np.array([cpc[x, y] for cpc in conn_PP_corr_c_list])
    fine_target = conn_PP_corr_f[x * renorm, y * renorm]
    ppcorr_f = np.ones_like(ppcorr_c) * fine_target
    ppcorr_c_err = _clean_err_list([em[x, y] if em.size else None for em in conn_PP_corr_c_err_list])
    fine_target_err = (float(conn_PP_corr_f_err[x * renorm, y * renorm])
                       if conn_PP_corr_f_err.size else None)

    # Error-weighted smoothing-spline fit through ALL scan points
    beta1_pp, beta1_pp_err = match_coupling(
        data_beta1_c_list, ppcorr_c, fine_target,
        y_err=ppcorr_c_err, y_target_err=fine_target_err)
    key = f'({x},{y})'
    results_pp[key] = beta1_pp
    results_pp_err[key] = beta1_pp_err

    add_coarse_points(data_beta1_c_list, ppcorr_c, ppcorr_c_err, label=f'Coarse ({x},{y})')
    plt.plot(data_beta1_c_list, ppcorr_f, label=f'Fine ({x*renorm},{y*renorm})')
    add_fit_curve(data_beta1_c_list, ppcorr_c, ppcorr_c_err)
    plt.xlabel('Beta1')
    plt.ylabel('PP Correlation')
    plt.legend()
    plt.title(f'beta_c = {beta_c:.3f}, alpha_c = {alpha_c:.3f}')

    if beta1_pp is not None:
        y_min, y_max = plt.ylim()
        plt.axvline(x=beta1_pp, color='red', linestyle='--', alpha=0.7)
        plt.text(beta1_pp, y_min + 0.95 * (y_max - y_min),
                 f'  beta1 = {_fmt_match(beta1_pp, beta1_pp_err)}', color='red',
                 fontweight='bold', va='top')
        print(f"  PP corr ({x},{y}): fine = {fine_target:.6f}, matched beta1_c = {_fmt_match(beta1_pp, beta1_pp_err)}")
    else:
        print(f"  PP corr ({x},{y}): fine = {fine_target:.6f}, NO MATCH (outside coarse range)")

    file_name = f"PP_corr_({x},{y}).png"
    file_path = os.path.join(folder_path, file_name)
    plt.savefig(file_path)

    plt.show()

In [ ]:
def corr_len(conn_PP_corr):
    L = conn_PP_corr.shape[0]
    PP_corr_k = np.fft.fftn(conn_PP_corr).real
    xi_x = 0.5 * np.sqrt(PP_corr_k[0,0] / PP_corr_k[1,0] - 1) / np.sin(np.pi / L)
    xi_y = 0.5 * np.sqrt(PP_corr_k[0,0] / PP_corr_k[0,1] - 1) / np.sin(np.pi / L)
    return (xi_x + xi_y) / 2

corrlen_c = np.array([corr_len(conn_PP_corr_c) for conn_PP_corr_c in conn_PP_corr_c_list]) * renorm
fine_target = corr_len(conn_PP_corr_f)
corrlen_f = np.ones_like(corrlen_c) * fine_target

# corr_len is FFT-derived -> no per-point error bars; fit unweighted.
beta1_pp, beta1_pp_err = match_coupling(data_beta1_c_list, corrlen_c, fine_target)
key = 'corr_len'
results_pp[key] = beta1_pp
results_pp_err[key] = beta1_pp_err

add_coarse_points(data_beta1_c_list, corrlen_c, None, label=f'Coarse length * {renorm}')
plt.plot(data_beta1_c_list, corrlen_f, label=f'Fine length')
add_fit_curve(data_beta1_c_list, corrlen_c)
plt.xlabel('Beta1')
plt.ylabel('PP Correlation Length')
plt.legend()
plt.title(f'beta_c = {beta_c:.3f}, alpha_c = {alpha_c:.3f}')

if beta1_pp is not None:
    y_min, y_max = plt.ylim()
    plt.axvline(x=beta1_pp, color='red', linestyle='--', alpha=0.7)
    plt.text(beta1_pp, y_min + 0.95 * (y_max - y_min),
                f'  beta1 = {_fmt_match(beta1_pp, beta1_pp_err)}', color='red',
                fontweight='bold', va='top')
    print(f"  PP corr len: fine = {fine_target:.6f}, matched beta1_c = {_fmt_match(beta1_pp, beta1_pp_err)}")
else:
    print(f"  PP corr len: fine = {fine_target:.6f}, NO MATCH (outside coarse range)")

file_name = f"PP_corr_len.png"
file_path = os.path.join(folder_path, file_name)
plt.savefig(file_path)

plt.show()

### Compare Magnetic Susceptibility ($\mathrm{mag\_sus} = \sum \mathrm{conn\_PP\_corr}$)

In [ ]:
# Magnetic susceptibility: mag_sus = sum(conn_PP_corr). Under blocking by factor
# `renorm` the coarse susceptibility carries an extra L**2 factor, so compare
# mag_sus_fine against mag_sus_coarse * renorm**2.
magsus_c = np.array([np.sum(cpc) for cpc in conn_PP_corr_c_list]) * renorm ** 2
fine_target = float(np.sum(conn_PP_corr_f))
magsus_f = np.ones_like(magsus_c) * fine_target
# error on sum(conn_PP_corr): sqrt(sum of variances) (independence approx), scaled
# by renorm**2 just like mag_sus itself.
magsus_c_err = _clean_err_list([np.sqrt(np.sum(em ** 2)) if em.size else None
                                for em in conn_PP_corr_c_err_list])
if magsus_c_err is not None:
    magsus_c_err = [e * renorm ** 2 for e in magsus_c_err]
fine_target_err = (float(np.sqrt(np.sum(conn_PP_corr_f_err ** 2)))
                   if conn_PP_corr_f_err.size else None)

# Error-weighted smoothing-spline fit through ALL scan points
beta1_magsus, beta1_magsus_err = match_coupling(
    data_beta1_c_list, magsus_c, fine_target,
    y_err=magsus_c_err, y_target_err=fine_target_err)
key = 'magsus'
results_pp[key] = beta1_magsus
results_pp_err[key] = beta1_magsus_err

add_coarse_points(data_beta1_c_list, magsus_c, magsus_c_err, label=f'Coarse mag_sus * {renorm}**2')
plt.plot(data_beta1_c_list, magsus_f, label=f'Fine mag_sus')
add_fit_curve(data_beta1_c_list, magsus_c, magsus_c_err)
plt.xlabel('Beta1')
plt.ylabel('Magnetic Susceptibility')
plt.legend()
plt.title(f'beta_c = {beta_c:.3f}, alpha_c = {alpha_c:.3f}')

if beta1_magsus is not None:
    y_min, y_max = plt.ylim()
    plt.axvline(x=beta1_magsus, color='red', linestyle='--', alpha=0.7)
    plt.text(beta1_magsus, y_min + 0.95 * (y_max - y_min),
                f'  beta1 = {_fmt_match(beta1_magsus, beta1_magsus_err)}', color='red',
                fontweight='bold', va='top')
    print(f"  mag_sus: fine = {fine_target:.6f}, matched beta1_c = {_fmt_match(beta1_magsus, beta1_magsus_err)}")
else:
    print(f"  mag_sus: fine = {fine_target:.6f}, NO MATCH (outside coarse range)")

file_name = f"mag_sus.png"
file_path = os.path.join(folder_path, file_name)
plt.savefig(file_path)

plt.show()

### Compare Topological Susceptibility ($\mathrm{topo\_sus} = (\langle Q^2\rangle - \langle Q\rangle^2)/V$, RefVil only)

In [ ]:
# Topological susceptibility (RefVil only). topo_sus = (<Q^2> - <Q>^2)/V is stored
# per component in topo_charge. Coarse uses the integer-winding Q_s susceptibility
# (unscaled); the FINE value is scaled by L**2 before matching. The fine component
# depends on renorm_type (z -> Q_z, U -> Q_s). Skipped if Q_s is unavailable.
fine_comp = 'Q_z' if renorm_type == 'z' else 'Q_s'
coarse_has_s = all('Q_s' in tc for tc in topo_charge_c_list)
if coarse_has_s and fine_comp in topo_charge_f and 'topo_sus' in topo_charge_f[fine_comp]:
    toposus_c = np.array([tc['Q_s']['topo_sus'] for tc in topo_charge_c_list])
    fine_raw = float(topo_charge_f[fine_comp]['topo_sus'])
    fine_target = fine_raw * renorm ** 2          # fine scaled by L**2
    toposus_f = np.ones_like(toposus_c) * fine_target
    toposus_c_err = _clean_err_list([tc['Q_s'].get('err') for tc in topo_charge_c_list])
    ferr = topo_charge_f[fine_comp].get('err')
    fine_target_err = float(ferr) * renorm ** 2 if ferr is not None else None

    # Error-weighted smoothing-spline fit through ALL scan points
    beta1_toposus, beta1_toposus_err = match_coupling(
        data_beta1_c_list, toposus_c, fine_target,
        y_err=toposus_c_err, y_target_err=fine_target_err)
    key = 'toposus'
    results_pp[key] = beta1_toposus
    results_pp_err[key] = beta1_toposus_err

    add_coarse_points(data_beta1_c_list, toposus_c, toposus_c_err, label=f'Coarse topo_sus(Q_s)')
    plt.plot(data_beta1_c_list, toposus_f, label=f'Fine topo_sus({fine_comp}) * {renorm}**2')
    add_fit_curve(data_beta1_c_list, toposus_c, toposus_c_err)
    plt.xlabel('Beta1')
    plt.ylabel('Topological Susceptibility')
    plt.legend()
    plt.title(f'beta_c = {beta_c:.3f}, alpha_c = {alpha_c:.3f}')

    if beta1_toposus is not None:
        y_min, y_max = plt.ylim()
        plt.axvline(x=beta1_toposus, color='red', linestyle='--', alpha=0.7)
        plt.text(beta1_toposus, y_min + 0.95 * (y_max - y_min),
                    f'  beta1 = {_fmt_match(beta1_toposus, beta1_toposus_err)}', color='red',
                    fontweight='bold', va='top')
        print(f"  topo_sus: fine({fine_comp})={fine_raw:.6f}*{renorm}**2={fine_target:.6f}, "
              f"matched beta1_c = {_fmt_match(beta1_toposus, beta1_toposus_err)}")
    else:
        print(f"  topo_sus: fine({fine_comp})={fine_raw:.6f}*{renorm}**2={fine_target:.6f}, "
              f"NO MATCH (outside coarse range)")

    file_name = f"topo_sus.png"
    file_path = os.path.join(folder_path, file_name)
    plt.savefig(file_path)
    plt.show()
else:
    print(f"  topo_sus: Q_s unavailable (obs_fit_type={obs_fit_type}); skipped")

### Compare $\arg(\bar z'z)$ Loop

In [ ]:
for xy in argzz_xy_list:
    x, y = int(xy[0]), int(xy[1])
    kc, kf = f'{x}x{y}', f'{x*renorm}x{y*renorm}'
    if not all(kc in wl for wl in argzz_loop_c_list) or kf not in argzz_loop_f:
        print(f"  argzz loop ({x},{y}): not measured for this size, skip")
        continue
    argzz_loop_c = np.array([wl[kc]['mean'] for wl in argzz_loop_c_list])
    fine_target = argzz_loop_f[kf]['mean']
    argzz_loop_f_plot = np.ones_like(argzz_loop_c) * fine_target
    argzz_c_err = _clean_err_list([wl[kc].get('err') for wl in argzz_loop_c_list])
    fine_target_err = argzz_loop_f[kf].get('err')

    # Error-weighted smoothing-spline fit through ALL scan points
    beta1_argzz, beta1_argzz_err = match_coupling(
        data_beta1_c_list, argzz_loop_c, fine_target,
        y_err=argzz_c_err, y_target_err=fine_target_err)
    key = f'({x},{y})'
    results_argzz[key] = beta1_argzz
    results_argzz_err[key] = beta1_argzz_err

    add_coarse_points(data_beta1_c_list, argzz_loop_c, argzz_c_err, label=f'coarse ({x},{y})')
    plt.plot(data_beta1_c_list, argzz_loop_f_plot, label=f'fine ({x*renorm},{y*renorm})')
    add_fit_curve(data_beta1_c_list, argzz_loop_c, argzz_c_err)
    plt.xlabel('Beta1')
    plt.ylabel('Arg(zz) Loop')
    plt.legend()
    plt.title(f'beta_c = {beta_c:.3f}, alpha_c = {alpha_c:.3f}')

    if beta1_argzz is not None:
        y_min, y_max = plt.ylim()
        plt.axvline(x=beta1_argzz, color='red', linestyle='--', alpha=0.7)
        plt.text(beta1_argzz, y_min + 0.95 * (y_max - y_min),
                 f'  beta1 = {_fmt_match(beta1_argzz, beta1_argzz_err)}', color='red',
                 fontweight='bold', va='top')
        print(f"  argzz loop ({x},{y}): fine = {fine_target:.6f}, matched beta1_c = {_fmt_match(beta1_argzz, beta1_argzz_err)}")
    else:
        print(f"  argzz loop ({x},{y}): fine = {fine_target:.6f}, NO MATCH (outside coarse range)")

    file_name = f"argzz_loop_({x},{y}).png"
    file_path = os.path.join(folder_path, file_name)
    plt.savefig(file_path)

    plt.show()

### Compare Wilson Loop

In [ ]:
idx = 0 if renorm_type == 'U' else 1
wl_label = 'Wilson(U)' if renorm_type == 'U' else 'Wilson(z)'

for xy in wilson_xy_list:
    x, y = int(xy[0]), int(xy[1])
    kc, kf = f'{x}x{y}', f'{x*renorm}x{y*renorm}'
    if not all(kc in wl for wl in wilson_loop_c_list) or kf not in wilson_loop_f:
        print(f"  {wl_label} ({x},{y}): not measured for this size, skip")
        continue
    wilson_loop_c_plot = np.array([wl[kc]['mean'][0] for wl in wilson_loop_c_list])
    fine_target = wilson_loop_f[kf]['mean'][idx]
    wilson_loop_f_plot = np.ones_like(wilson_loop_c_plot) * fine_target
    wl_c_err = _clean_err_list([wl[kc]['err'][0] for wl in wilson_loop_c_list])
    fine_target_err = wilson_loop_f[kf]['err'][idx]

    # Error-weighted smoothing-spline fit through ALL scan points
    beta1_wl, beta1_wl_err = match_coupling(
        data_beta1_c_list, wilson_loop_c_plot, fine_target,
        y_err=wl_c_err, y_target_err=fine_target_err)
    key = f'({x},{y})'
    results_wilson[key] = beta1_wl
    results_wilson_err[key] = beta1_wl_err

    add_coarse_points(data_beta1_c_list, wilson_loop_c_plot, wl_c_err, label=f'coarse ({x},{y})')
    plt.plot(data_beta1_c_list, wilson_loop_f_plot, label=f'fine ({x*renorm},{y*renorm})')
    add_fit_curve(data_beta1_c_list, wilson_loop_c_plot, wl_c_err)
    plt.xlabel('Beta1')
    plt.ylabel('Wilson Loop')
    plt.legend()
    plt.title(f'beta_c = {beta_c:.3f}, alpha_c = {alpha_c:.3f}')

    if beta1_wl is not None:
        y_min, y_max = plt.ylim()
        plt.axvline(x=beta1_wl, color='red', linestyle='--', alpha=0.7)
        plt.text(beta1_wl, y_min + 0.95 * (y_max - y_min),
                 f'  beta1 = {_fmt_match(beta1_wl, beta1_wl_err)}', color='red',
                 fontweight='bold', va='top')
        print(f"  {wl_label} ({x},{y}): fine = {fine_target:.6f}, matched beta1_c = {_fmt_match(beta1_wl, beta1_wl_err)}")
    else:
        print(f"  {wl_label} ({x},{y}): fine = {fine_target:.6f}, NO MATCH (outside coarse range)")

    file_name = f"wilson_loop_({x},{y}).png"
    file_path = os.path.join(folder_path, file_name)
    plt.savefig(file_path)

    plt.show()

### Summary: Averaged beta1_c Estimates

In [ ]:
print("=" * 70)
print("SUMMARY: Matched beta1_c by error-weighted smoothing spline")
print(f"renorm = {renorm}, type = '{renorm_type}', obs_fit_type = '{obs_fit_type}'")
print(f"Fine:   L={L_f}, beta={beta_f}, beta1={beta1_f}, alpha={alpha_f}, alpha1={alpha1_f}")
print(f"Coarse: L={L_c}, beta={beta_c}, alpha={alpha_c}, alpha1={alpha1_c}")
print("=" * 70)

# Single global average over all matched beta1 across pp + argzz + wilson.
# corr_len, magsus, toposus are excluded (kept as their own outputs in results_pp).
collected = []
for result_dict, err_dict, label in [
    (results_pp, results_pp_err, "PP corr"),
    (results_argzz, results_argzz_err, "argzz"),
    (results_wilson, results_wilson_err, "Wilson"),
]:
    for key, v in result_dict.items():
        if key in ("corr_len", "magsus", "toposus") or v is None:
            continue
        collected.append(v)
        print(f"  {label} {key}: beta1_c = {_fmt_match(v, err_dict.get(key))}")

if collected:
    avg = float(np.mean(collected))
    std = float(np.std(collected)) if len(collected) > 1 else 0.0
    print(f"\n  >>> GLOBAL AVERAGE beta1_c = {avg:.4f} +/- {std:.4f}  (n={len(collected)})")
else:
    avg, std = None, None
    print("\n  >>> NO MATCH for any observable")

# corr_len / magsus / toposus are their own beta1_c estimates (excluded from avg).
summary = {
    "avg": avg, "std": std, "n": len(collected),
    "corr": results_pp.get("corr_len"),      "corr_err": results_pp_err.get("corr_len"),
    "magsus": results_pp.get("magsus"),      "magsus_err": results_pp_err.get("magsus"),
    "toposus": results_pp.get("toposus"),    "toposus_err": results_pp_err.get("toposus"),
}

# Save results to file
results_dict = {
    'N': N, 'L_f': L_f, 'L_c': L_c, 'renorm': renorm, 'renorm_type': renorm_type,
    'obs_fit_type': obs_fit_type,
    'beta_f': beta_f, 'beta1_f': beta1_f, 'alpha_f': alpha_f, 'alpha1_f': alpha1_f,
    'beta_c': beta_c, 'alpha_c': alpha_c, 'alpha1_c': alpha1_c,
    'beta1_c_pp': {k: v for k, v in results_pp.items()},
    'beta1_c_pp_err': {k: v for k, v in results_pp_err.items()},
    'beta1_c_argzz': {k: v for k, v in results_argzz.items()},
    'beta1_c_argzz_err': {k: v for k, v in results_argzz_err.items()},
    'beta1_c_wilson': {k: v for k, v in results_wilson.items()},
    'beta1_c_wilson_err': {k: v for k, v in results_wilson_err.items()},
    "summary": summary,
}
results_file = os.path.join(folder_path, "matching_results.json")
with open(results_file, 'w') as f:
    json.dump(results_dict, f, indent=4, default=str)
print(f"\nResults saved to {results_file}")